### Last Claude Chat - Linear layer from scratch

In [109]:
def contiguous_strides(shape):
    res = []
    p = 1
    for s in reversed(shape):
        res.append(p)
        p *= s
    return tuple(reversed(res))


def infer_shape(nested):
    shape = []
    while isinstance(nested, list):
        shape.append(len(nested))
        if len(nested) == 0:
            break
        nested = nested[0]  # descend into the first element
    return tuple(shape)


def flatten(nested):
    if not isinstance(nested, list):
        return [nested]
    out = []
    for e in nested:
        out += flatten(e)
    return out


def prod(xs):
    out = 1
    for x in xs:
        out *= x
    return out


class Tensor:
    def __init__(self, data, shape=None, strides=None):
        if shape is None:
            shape = infer_shape(data)
            data = flatten(data)
        self.data = data
        self.shape = shape
        self.strides = contiguous_strides(shape) if strides is None else strides

    def _offset(self, idx):
        assert len(idx) == len(self.shape)
        return sum(i * o for i, o in zip(idx, self.strides))

    def transpose(self, d0, d1):
        shape, strides = list(self.shape), list(self.strides)
        shape[d0], shape[d1] = shape[d1], shape[d0]
        strides[d0], strides[d1] = strides[d1], strides[d0]
        return Tensor(self.data, tuple(shape), tuple(strides))

    def tolist(self):
        def build(idx):
            if len(idx) == len(self.shape):
                return self.data[self._offset(idx)]
            d = len(idx)
            return [build(idx + (i,)) for i in range(self.shape[d])]

        return build(())

    @property
    def numel(self):
        return prod(self.shape)

    def is_contiguous(self):
        return self.strides == contiguous_strides(self.shape)

    def flat(self):
        """values in logical (row-major) order, respecting strides"""
        return flatten(self.tolist())

    def contiguous(self):
        if self.is_contiguous():
            return self
        return Tensor(self.flat(), self.shape)

    def __repr__(self):
        return (
            f"Tensor(shape={self.shape}, strides={self.strides}, data={self.tolist()})"
        )


In [110]:
a = Tensor([[1, 2, 3], [4, 5, 6]])
t = a.transpose(0, 1)

print(a.is_contiguous())   # True
print(a.tolist())
print(t.is_contiguous())   # False
print(t.data)              # [1, 2, 3, 4, 5, 6]  <- storage order
print(t.tolist())          # [[1, 4], [2, 5], [3, 6]]  <- logical order

True
[[1, 2, 3], [4, 5, 6]]
False
[1, 2, 3, 4, 5, 6]
[[1, 4], [2, 5], [3, 6]]


In [111]:
a = Tensor([[1, 2, 3], [4, 5, 6]])
a = a.transpose(0, 1)
print(repr(a))
t = a.contiguous()
print(repr(t))

Tensor(shape=(3, 2), strides=(1, 3), data=[[1, 4], [2, 5], [3, 6]])
Tensor(shape=(3, 2), strides=(2, 1), data=[[1, 4], [2, 5], [3, 6]])
